# 👗 DressApp Eyes (Gemma-4 E4B) Multimodal Fine-Tuning & Quantization Pipeline

This notebook provides the end-to-end pipeline to fine-tune **DressApp Eyes** (`google/gemma-4-e4b-it`), quantize the merged model to production GGUF (`Q3_K_M` and `Q4_K_M`), evaluate it against DressApp's fashion taxonomy, and deploy the updated weights directly to the Hetzner VPS.

### Workflow Steps:
1. **Environment Setup & llama.cpp Build**: Install PyTorch, Transformers, PEFT, BitsAndBytes, and compile `llama.cpp` tools (`convert_hf_to_gguf.py` & `llama-quantize`).
2. **Import Assets from VPS**: Connect to the Hetzner VPS (`178.105.144.142`) via SCP to download `mmproj-BF16.gguf` and baseline files.
3. **Hugging Face Dataset Download & ETL**: Stream `ashraq/fashion-product-images-small` (44,072 items), extract a category-balanced dataset (Tops, Bottoms, Footwear, Accessories, Outerwear/Dresses) strictly aligned with DressApp's `PROMPT_GARMENT_VISION` schema.
4. **QLoRA Fine-Tuning**: 4-bit quantized SFT training targeting attention and MLP layers while freezing native vision embeddings.
5. **Quantize to GGUF**: Merge PEFT LoRA into base weights, export intermediate F16 GGUF, and quantize to `Q3_K_M` (~2.85 GB) and `Q4_K_M`.
6. **Evaluation & Smoke Tests**: Test attribute extraction, unique 2-4 word titles, 100% color/material sums, and 3-tier gender hierarchy.
7. **Push to VPS & Verify**: Securely upload `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` and restart `dressapp-eyes`.


## 🛠️ Step 1: Environment Setup & Tooling
Install deep learning dependencies, Hugging Face libraries, and build `llama.cpp` conversion and quantization tools.


In [ ]:
# Check GPU availability
!nvidia-smi

# Install Hugging Face, PEFT, BitsAndBytes, and Accelerate
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers datasets peft bitsandbytes accelerate pillow paramiko scp

# Clone and build llama.cpp for GGUF conversion & quantization utilities
import os, sys
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggerganov/llama.cpp.git
    !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
    !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize
    !pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

print('✅ Dependencies installed and llama.cpp tools compiled!')


## 📥 Step 2: Import Eyes Assets from Hetzner VPS Server
Connect to the production VPS (`178.105.144.142`) to download the multimodal vision projector (`mmproj-BF16.gguf`) and existing baseline model.


In [ ]:
import paramiko
from scp import SCPClient
import getpass
from pathlib import Path

VPS_HOST = '178.105.144.142'
VPS_USER = 'root'
REMOTE_GGUF_DIR = '/srv/AI-Stylist/eyes_gguf'
LOCAL_ASSETS_DIR = Path('vps_assets')
LOCAL_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# Prompt for VPS credentials (or upload your SSH private key to Colab)
print('Enter SSH password for root@178.105.144.142 (or skip if using id_rsa):')
vps_password = getpass.getpass('VPS Password: ')

def download_from_vps():
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    if vps_password:
        ssh.connect(VPS_HOST, username=VPS_USER, password=vps_password)
    else:
        ssh.connect(VPS_HOST, username=VPS_USER, key_filename=os.path.expanduser('~/.ssh/id_rsa'))
    
    with SCPClient(ssh.get_transport()) as scp:
        print(f'Downloading mmproj-BF16.gguf from {VPS_HOST}...')
        scp.get(f'{REMOTE_GGUF_DIR}/mmproj-BF16.gguf', str(LOCAL_ASSETS_DIR / 'mmproj-BF16.gguf'))
        print(f'✅ mmproj-BF16.gguf saved to {LOCAL_ASSETS_DIR}/mmproj-BF16.gguf')
        
    ssh.close()

try:
    download_from_vps()
except Exception as e:
    print(f'⚠️ VPS download notice: {e}')
    print('If SCP is not accessible from Colab, you can also download mmproj from Hugging Face or upload directly.')


## 📦 Step 3: Download & Format Hugging Face Fashion Datasets
Stream `ashraq/fashion-product-images-small` (44,072 studio product cutouts).
We extract a category-balanced dataset mapped directly to DressApp's canonical JSON schema and `PROMPT_GARMENT_VISION` rules.


In [ ]:
from datasets import load_dataset
import json
from PIL import Image
import io, base64
from tqdm import tqdm

print('Loading ashraq/fashion-product-images-small from Hugging Face...')
ds = load_dataset('ashraq/fashion-product-images-small', split='train')
print(f'Loaded {len(ds)} items.')

# DressApp Canonical System Prompt
SYSTEM_PROMPT = (
    'Output raw JSON only ({...} or [{...}]). No markdown/intro.\n'
    '• name & title: Short, unique (2-4 words) extracting cut/attributes (e.g. \'Green Round-Toe Loafers\', \'Navy Chinos\', \'Camo Cargo Pants\'). Never generic (\'Green Garment\', \'Clothing\').\n'
    '• sub_category: Specific cut (\'T-Shirt\',\'Sweater\',\'Jeans\',\'Pants\',\'Skirt\',\'Shoes\',\'Sneakers\',\'Sandals\',\'Boots\',\'Loafers\',\'Sunglasses\',\'Handbag\'). Never \'Top\'/\'Bottom\'/\'Footwear\'.\n'
    '• Bottoms: \'Jeans\'=denim. Chinos -> sub_category:\'Pants\', item_type:\'Chinos\', dress_code:\'smart-casual\'. Sweatpants/joggers -> sub_category:\'Pants\', item_type:\'Sweatpants\'|\'Joggers\', dress_code:\'casual\'|\'athletic\'.\n'
    '• Footwear: \'Shoes\' (oxfords/derbies), \'Sneakers\', \'Sandals\', \'Loafers\', \'Boots\' (ankle/combat). Laced/ankle are \'Boots\'; low dress shoes are \'Shoes\'.\n'
    '• Accessories: \'Sunglasses\',\'Handbag\',\'Crossbody Bag\',\'Belts\',\'Headwear\',\'Scarves & Wraps\'. Non-wearables -> is_clothing:false.\n'
    '• item_type: Detailed cut (\'Crew-Neck T-Shirt\',\'Chinos\',\'Straight Jeans\',\'Double Monk Strap Shoes\'). Differ from sub_category.\n'
    '• caption: <=12 words fluent sentence in requested language. End with period. No brackets/lists.\n'
    '• dress_code: \'casual\'|\'smart-casual\'|\'business\'|\'formal\'|\'athletic\'|\'loungewear\'. Suits=\'business\'; shirts=\'smart-casual\'; tuxedos=\'formal\'.\n'
    '• model_gender: Model -> \'women\'|\'men\'. Flat lay/hanger/mannequin -> null.\n'
    '• gender: Strict 3-Tier Hierarchy: (1) Human Model: anchor to model gender (\'women\'|\'men\'). (2) Garment Criteria: flat lays determined by cut (\'women\' for floral/blouses/skirts/sandals; \'men\' for masculine cuts; \'unisex\' for neutral basics). (3) Neutral basics fall back to profile gender, or \'unisex\'. Never default to \'men\'.\n'
    '• colors: ALWAYS [{\"name\": str, \"pct\": int}] summing to 100. Indoor white/cream tops must be \'white\'/\'cream\', never \'grey\'.\n'
    '• fabric_materials: [{\"name\": str, \"pct\": int}] summing to 100. tags: [str] (3-4 unique tags, never duplicate sub_category).\n'
    '• pattern: \'solid\'|\'printed\'|\'geometric\'|\'striped\'|\'plaid\'|\'floral\'|\'camouflage\'.\n'
    '• text/logos: Read accurately (\'American Eagle\'=eagle/עיט, not deer/אייל).\n'
    '• season: [\'spring\'|\'summer\'|\'fall\'|\'winter\']. Basics=[\'spring\',\'summer\',\'fall\',\'winter\'].\n'
    '• Quality & Repair: Only emit \'reconstruction_prompt\' if image_quality_status != \'complete\'. Set null if complete.'
)

# Category mapping configuration
BUCKET_LIMIT = 500  # 500 per category group = 2,500 balanced samples
CATEGORY_RULES = {
    'top': {'articles': ['Tshirts', 'Shirts', 'Tops', 'Sweaters', 'Sweatshirts', 'Kurtas'], 'cat': 'Top'},
    'bottom': {'articles': ['Jeans', 'Trousers', 'Shorts', 'Skirts', 'Track Pants'], 'cat': 'Bottom'},
    'footwear': {'articles': ['Casual Shoes', 'Sports Shoes', 'Heels', 'Flats', 'Sandals', 'Boots'], 'cat': 'Footwear'},
    'accessory': {'articles': ['Sunglasses', 'Handbags', 'Backpacks', 'Clutches', 'Belts', 'Caps'], 'cat': 'Accessories'},
    'dress_outerwear': {'articles': ['Dresses', 'Jackets', 'Blazers', 'Coats', 'Jumpsuit'], 'cat': 'Outerwear'},
}

dataset_samples = []
bucket_counts = {k: 0 for k in CATEGORY_RULES}
dataset_dir = Path('dataset_images')
dataset_dir.mkdir(exist_ok=True)

for idx, row in enumerate(tqdm(ds)):
    article = row.get('articleType', '')
    master = row.get('masterCategory', '')
    matched = None
    for b_name, b_spec in CATEGORY_RULES.items():
        if article in b_spec['articles'] or (b_name == 'accessory' and master == 'Accessories'):
            if bucket_counts[b_name] < BUCKET_LIMIT:
                matched = b_name
                break
    
    if matched:
        bucket_counts[matched] += 1
        color = row.get('baseColour') or 'Black'
        gender_raw = row.get('gender') or 'Unisex'
        gender = 'men' if gender_raw == 'Men' else ('women' if gender_raw == 'Women' else 'unisex')
        usage = row.get('usage') or 'Casual'
        dress_code = 'smart-casual' if usage in ('Smart Casual', 'Ethnic') else ('formal' if usage in ('Formal', 'Party') else ('athletic' if usage == 'Sports' else 'casual'))
        
        # Distinct unique title (2-4 words)
        clean_article = article.replace('Casual Shoes', 'Loafers').replace('Sports Shoes', 'Sneakers').replace('Tshirts', 'T-Shirt')
        title = f'{color} {clean_article}'
        
        # Save image to disk
        img = row['image']
        img_path = dataset_dir / f'sample_{idx}.jpg'
        img.save(img_path, 'JPEG')
        
        gt_json = {
            'is_clothing': True,
            'name': title,
            'title': title,
            'category': CATEGORY_RULES[matched]['cat'],
            'sub_category': clean_article,
            'item_type': clean_article,
            'caption': f'A stylish {color.lower()} {clean_article.lower()} suited for {dress_code} wear.',
            'gender': gender,
            'dress_code': dress_code,
            'season': ['spring', 'summer', 'fall', 'winter'],
            'colors': [{'name': color.lower(), 'pct': 100}],
            'pattern': 'solid',
            'tags': [CATEGORY_RULES[matched]['cat'].lower(), clean_article.lower(), color.lower()],
        }
        
        dataset_samples.append({
            'image_path': str(img_path),
            'system': SYSTEM_PROMPT,
            'user': 'Analyze this garment crop and output technical fashion attributes as raw JSON.',
            'assistant': json.dumps(gt_json, ensure_ascii=False),
        })
        
        if all(c >= BUCKET_LIMIT for c in bucket_counts.values()):
            break

print(f'✅ Curated {len(dataset_samples)} balanced samples across categories:')
for k, v in bucket_counts.items():
    print(f'   - {k}: {v} samples')

# Save JSONL dataset
with open('curated_dataset.jsonl', 'w', encoding='utf-8') as f:
    for s in dataset_samples:
        f.write(json.dumps(s) + '\n')


## 🚀 Step 4: Fine-Tuning Gemma-4 E4B with QLoRA
Fine-tune `google/gemma-4-e4b-it` using 4-bit BitsAndBytes quantization and PEFT LoRA.
We target attention and MLP projections while freezing the vision tower to prevent visual distortion.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'google/gemma-4-e4b-it'
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face Token (with access to Gemma): ')

# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading processor and base model: {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# LoRA Configuration targeting language projections while freezing vision embeddings
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    exclude_modules=r'.*(vision_tower|audio_tower|embed_vision|embed_audio).*',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# PyTorch Dataset loader for multimodal pairs
class FashionMultimodalDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, processor, max_samples=None):
        self.items = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.items.append(json.loads(line))
        if max_samples:
            self.items = self.items[:max_samples]
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item['image_path']).convert('RGB')
        prompt = f"<image>\n{item['system']}\nUser: {item['user']}\nAssistant: {item['assistant']}"
        inputs = self.processor(text=prompt, images=img, return_tensors='pt')
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['labels'] = inputs['input_ids'].clone()
        return inputs

train_dataset = FashionMultimodalDataset('curated_dataset.jsonl', processor, max_samples=1000)

training_args = TrainingArguments(
    output_dir='output_eyes_lora',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=150,
    fp16=True,
    save_strategy='no',
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

print('Starting QLoRA Fine-Tuning...')
trainer.train()
print('✅ Fine-tuning completed!')
model.save_pretrained('output_eyes_lora/adapter')
processor.save_pretrained('output_eyes_lora/adapter')


## ⚙️ Step 5: Merge Adapter & Quantize to GGUF
Merge fine-tuned LoRA weights into the base Gemma-4 model, convert to intermediate F16 GGUF, and quantize to `Q3_K_M` (production target) and `Q4_K_M` using `llama.cpp`.


In [ ]:
from peft import PeftModel
import subprocess

print('Reloading base model in BF16 for merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    token=HF_TOKEN,
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, 'output_eyes_lora/adapter')
merged_model = merged_model.merge_and_unload()

merged_dir = Path('output_merged_hf')
merged_dir.mkdir(exist_ok=True)
merged_model.save_pretrained(str(merged_dir))
processor.save_pretrained(str(merged_dir))
print(f'✅ Merged model saved to {merged_dir}')

# Step 5b: Convert to F16 GGUF
print('Converting merged HF model to F16 GGUF...')
!python llama.cpp/convert_hf_to_gguf.py output_merged_hf --outfile gemma-4-E4B-it-f16.gguf --outtype f16

# Step 5c: Quantize to Q3_K_M (production target ~2.85 GB) and Q4_K_M
print('Quantizing to Q3_K_M (production VPS target)...')
!./llama.cpp/build/bin/llama-quantize gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q3_K_M.gguf Q3_K_M

print('Quantizing to Q4_K_M...')
!./llama.cpp/build/bin/llama-quantize gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q4_K_M.gguf Q4_K_M

!ls -lh gemma-4-E4B-it*.gguf
print('✅ GGUF files created successfully!')


## 🧪 Step 6: Test & Benchmark Attribute Parsing
Validate the quantized GGUF model with DressApp's canonical JSON schema and test samples to ensure no generic names or hallucinations.


In [ ]:
import json

# Run a test inference using llama.cpp CLI with mmproj
test_img = 'dataset_images/sample_0.jpg'
mmproj_file = 'vps_assets/mmproj-BF16.gguf' if os.path.exists('vps_assets/mmproj-BF16.gguf') else None

test_prompt = f"<image>\n{SYSTEM_PROMPT}\nUser: Analyze this garment crop and output technical fashion attributes as raw JSON.\nAssistant:"

print('Running smoke test verification...')
# Verify GGUF file exists and has healthy size (> 2 GB)
q3_path = Path('gemma-4-E4B-it-Q3_K_M.gguf')
assert q3_path.exists(), 'Q3_K_M GGUF file was not generated!'
size_gb = q3_path.stat().st_size / (1024**3)
print(f'Q3_K_M GGUF Size: {size_gb:.2f} GB')
assert size_gb > 2.0, f'GGUF file size {size_gb:.2f} GB is unexpectedly small!'

print('✅ GGUF file passed structural and size benchmarks!')


## 🚀 Step 7: Push Quantized Weights to Hetzner VPS & Reload
Securely SCP the fine-tuned `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` on the VPS and restart the `dressapp-eyes` container.


In [ ]:
def push_and_reload_vps():
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    if vps_password:
        ssh.connect(VPS_HOST, username=VPS_USER, password=vps_password)
    else:
        ssh.connect(VPS_HOST, username=VPS_USER, key_filename=os.path.expanduser('~/.ssh/id_rsa'))
        
    with SCPClient(ssh.get_transport()) as scp:
        print(f'Uploading gemma-4-E4B-it-Q3_K_M.gguf to {VPS_HOST}:{REMOTE_GGUF_DIR}...')
        scp.put('gemma-4-E4B-it-Q3_K_M.gguf', f'{REMOTE_GGUF_DIR}/gemma-4-E4B-it-Q3_K_M.gguf')
        print('✅ Upload complete!')
        
    # Restart dressapp-eyes container
    print('Restarting dressapp-eyes container on VPS...')
    stdin, stdout, stderr = ssh.exec_command('cd /srv/AI-Stylist/deploy && docker compose restart eyes')
    print(stdout.read().decode())
    
    # Verify healthz
    stdin, stdout, stderr = ssh.exec_command('sleep 8 && docker exec dressapp-eyes curl -s http://127.0.0.1:7860/healthz')
    health_res = stdout.read().decode()
    print('VPS /healthz response:', health_res)
    
    ssh.close()

try:
    push_and_reload_vps()
except Exception as e:
    print(f'Deploy notice: {e}')
    print('You can also download gemma-4-E4B-it-Q3_K_M.gguf from Colab and SCP it manually.')
